# Management System API Testing

This notebook provides comprehensive testing for all REST API endpoints in the Management System backend.

## Table of Contents
1. [Setup & Configuration](#setup)
2. [Authentication APIs](#authentication)
3. [User Management APIs](#users)
4. [Employee Management APIs](#employees)
5. [Client Management APIs](#clients)
6. [Client Unit APIs](#client-units)
7. [Department Contact APIs](#department-contacts)

---

## 1. Setup & Configuration <a id="setup"></a>

Install required packages and configure the base URL and authentication.

In [13]:
# Install required packages (run once)
# !pip install requests python-dotenv

In [14]:
import requests
import json
from datetime import datetime, timedelta
from pprint import pprint

# Configuration
BASE_URL = "http://localhost:8000/api"  # Change this to your backend URL
HEADERS = {"Content-Type": "application/json"}

# Global variables to store tokens
ACCESS_TOKEN = None
REFRESH_TOKEN = None

# Helper function to update headers with token
def set_auth_token(token):
    global HEADERS
    HEADERS["Authorization"] = f"Bearer {token}"

# Helper function to print responses
def print_response(response, title="Response"):
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")
    print(f"Status Code: {response.status_code}")
    print(f"\nResponse Body:")
    try:
        pprint(response.json())
    except:
        print(response.text)
    print(f"{'='*60}\n")

print("✅ Setup complete!")
print(f"Base URL: {BASE_URL}")

✅ Setup complete!
Base URL: http://localhost:8000/api


---
## 2. Authentication APIs <a id="authentication"></a>

Test JWT authentication endpoints: login, token refresh, and token verification.

### 2.1 User Registration

Create a new user account for testing.

In [15]:
# Register a new user
register_data = {
    "name": "Test User",
    "username": "Test User",
    "email": "testuser@example.com",
    "password": "TestPassword123!",
    "password_confirm": "TestPassword123!"
}

response = requests.post(f"{BASE_URL}/users/", json=register_data, headers=HEADERS)
print_response(response, "User Registration")

if response.status_code == 201:
    user_data = response.json()
    USER_ID = user_data.get('id')
    print(f"✅ User created successfully with ID: {USER_ID}")
else:
    print("❌ Registration failed. You may use existing credentials for login.")


User Registration
Status Code: 400

Response Body:
{'email': ['User with this email already exists.']}

❌ Registration failed. You may use existing credentials for login.


### 2.2 Login (Obtain JWT Token)

Login with credentials to get access and refresh tokens.

In [29]:
# Login to get JWT tokens
login_data = {
    # "email": "testuser@example.com",
    "email": "testuser@example.com",
    "password": "TestPassword123!"
}

response = requests.post(f"{BASE_URL}/token/", json=login_data, headers=HEADERS)
print_response(response, "Login (Token Obtain)")

if response.status_code == 200:
    tokens = response.json()
    ACCESS_TOKEN = tokens.get('access')
    REFRESH_TOKEN = tokens.get('refresh')
    set_auth_token(ACCESS_TOKEN)
    print("✅ Login successful! Token set in headers.")
else:
    print(response.text)
    print("❌ Login failed!")


Login (Token Obtain)
Status Code: 401

Response Body:
{'detail': 'No active account found with the given credentials'}

{"detail":"No active account found with the given credentials"}
❌ Login failed!


### 2.3 Verify Token

Verify that the access token is valid.

In [17]:
# Verify the access token
verify_data = {"token": ACCESS_TOKEN}

response = requests.post(f"{BASE_URL}/token/verify/", json=verify_data, headers=HEADERS)
print_response(response, "Token Verification")

if response.status_code == 200:
    print("✅ Token is valid!")
else:
    print("❌ Token is invalid or expired!")


Token Verification
Status Code: 400

Response Body:
{'token': ['This field may not be null.']}

❌ Token is invalid or expired!


### 2.4 Refresh Token

Get a new access token using the refresh token.

In [18]:
# Refresh the access token
refresh_data = {"refresh": REFRESH_TOKEN}

response = requests.post(f"{BASE_URL}/token/refresh/", json=refresh_data, headers=HEADERS)
print_response(response, "Token Refresh")

if response.status_code == 200:
    new_tokens = response.json()
    ACCESS_TOKEN = new_tokens.get('access')
    set_auth_token(ACCESS_TOKEN)
    print("✅ Token refreshed successfully!")
else:
    print("❌ Token refresh failed!")


Token Refresh
Status Code: 400

Response Body:
{'refresh': ['This field may not be null.']}

❌ Token refresh failed!


---
## 3. User Management APIs <a id="users"></a>

Test all user management endpoints.

### 3.1 Get Current User Profile (Me)

Retrieve the authenticated user's profile.

In [19]:
# Get current user profile
response = requests.get(f"{BASE_URL}/users/me/", headers=HEADERS)
print_response(response, "Current User Profile")

if response.status_code == 200:
    print("✅ Profile retrieved successfully!")
else:
    print("❌ Failed to retrieve profile!")


Current User Profile
Status Code: 401

Response Body:
{'detail': 'Authentication credentials were not provided.'}

❌ Failed to retrieve profile!


### 3.2 List All Users

Get a list of all users (Admin only).

In [20]:
# List all users
response = requests.get(f"{BASE_URL}/users/", headers=HEADERS)
print_response(response, "List All Users")

if response.status_code == 200:
    users = response.json()
    print(f"✅ Found {len(users)} users")
else:
    print("❌ Failed to retrieve users (may require admin privileges)")


List All Users
Status Code: 401

Response Body:
{'detail': 'Authentication credentials were not provided.'}

❌ Failed to retrieve users (may require admin privileges)


### 3.3 Get User by ID

Retrieve details of a specific user.

In [21]:
# Get user by ID (replace with actual user ID)
user_id = 1  # Change this to a valid user ID

response = requests.get(f"{BASE_URL}/users/{user_id}/", headers=HEADERS)
print_response(response, f"Get User {user_id}")

if response.status_code == 200:
    print("✅ User details retrieved successfully!")
else:
    print("❌ Failed to retrieve user details!")


Get User 1
Status Code: 401

Response Body:
{'detail': 'Authentication credentials were not provided.'}

❌ Failed to retrieve user details!


### 3.4 Update User

Update user information (partial update).

In [22]:
# Update user (PATCH request)
user_id = 1  # Change this to a valid user ID
update_data = {
    "name": "Updated Test User"
}

response = requests.patch(f"{BASE_URL}/users/{user_id}/", json=update_data, headers=HEADERS)
print_response(response, f"Update User {user_id}")

if response.status_code == 200:
    print("✅ User updated successfully!")
else:
    print("❌ Failed to update user!")


Update User 1
Status Code: 401

Response Body:
{'detail': 'Authentication credentials were not provided.'}

❌ Failed to update user!


### 3.5 Change Password

Change the password for the authenticated user.

In [23]:
# Change password
password_data = {
    "old_password": "TestPassword123!",
    "new_password": "NewPassword123!",
    "new_password2": "NewPassword123!"
}

response = requests.post(f"{BASE_URL}/users/change_password/", json=password_data, headers=HEADERS)
print_response(response, "Change Password")

if response.status_code == 200:
    print("✅ Password changed successfully!")
else:
    print("❌ Failed to change password!")


Change Password
Status Code: 401

Response Body:
{'detail': 'Authentication credentials were not provided.'}

❌ Failed to change password!


### 3.6 User Statistics

Get user statistics (Admin only).

In [24]:
# Get user statistics
response = requests.get(f"{BASE_URL}/users/statistics/", headers=HEADERS)
print_response(response, "User Statistics")

if response.status_code == 200:
    print("✅ Statistics retrieved successfully!")
else:
    print("❌ Failed to retrieve statistics (may require admin privileges)!")


User Statistics
Status Code: 401

Response Body:
{'detail': 'Authentication credentials were not provided.'}

❌ Failed to retrieve statistics (may require admin privileges)!


### 3.7 Delete User

Delete a user (Admin only).

In [25]:
# Delete user (Admin only - use with caution!)
# user_id = 999  # Change this to a valid user ID you want to delete

# response = requests.delete(f"{BASE_URL}/users/{user_id}/", headers=HEADERS)
# print_response(response, f"Delete User {user_id}")

# if response.status_code == 204:
#     print("✅ User deleted successfully!")
# else:
#     print("❌ Failed to delete user!")

print("⚠️ Delete user endpoint commented out for safety. Uncomment to test.")

⚠️ Delete user endpoint commented out for safety. Uncomment to test.


---
## 4. Employee Management APIs <a id="employees"></a>

Test all employee management endpoints.

### 4.1 Create Employee

Create a new employee record.

In [26]:
# Create a new employee
employee_data = {
    "employeeCode": "EMP001",
    "firstName": "John",
    "middleName": "Michael",
    "lastName": "Doe",
    "fatherName": "Robert Doe",
    "motherName": "Mary Doe",
    "dob": "1990-01-15T00:00:00Z",
    "gender": "Male",
    "maritalStatus": True,
    "spouseName": "Jane Doe",
    "email": "john.doe@example.com",
    "phone": "+1234567890",
    "mobileHome": "+0987654321",
    "position": "Software Engineer",
    "department": "IT",
    "hireDate": "2020-01-01",
    "salary": "75000.00",
    "height": "180cm",
    "weight": "75kg",
    "bloodGroup": "O+",
    "presentStreet": "123 Main St",
    "presentCity": "New York",
    "presentState": "NY",
    "permanentStreet": "123 Main St",
    "permanentCity": "New York",
    "permanentState": "NY",
    "aadharNumber": "1234-5678-9012",
    "panNumber": "ABCDE1234F"
}

response = requests.post(f"{BASE_URL}/employee/", json=employee_data, headers=HEADERS)
print_response(response, "Create Employee")

if response.status_code == 201:
    employee = response.json()
    EMPLOYEE_ID = employee.get('id')
    print(f"✅ Employee created successfully with ID: {EMPLOYEE_ID}")
else:
    print("❌ Failed to create employee!")


Create Employee
Status Code: 401

Response Body:
{'detail': 'Authentication credentials were not provided.'}

❌ Failed to create employee!


### 4.2 List All Employees

Get a list of all employees.

In [27]:
# List all employees
response = requests.get(f"{BASE_URL}/employee/", headers=HEADERS)
print_response(response, "List All Employees")

if response.status_code == 200:
    employees = response.json()
    print(f"✅ Found {len(employees)} employees")
else:
    print("❌ Failed to retrieve employees!")


List All Employees
Status Code: 401

Response Body:
{'detail': 'Authentication credentials were not provided.'}

❌ Failed to retrieve employees!


### 4.3 Get Employee by ID

Retrieve details of a specific employee.

In [28]:
# Get employee by ID
employee_id = 1  # Change this to a valid employee ID

response = requests.get(f"{BASE_URL}/employee/{employee_id}/", headers=HEADERS)
print_response(response, f"Get Employee {employee_id}")

if response.status_code == 200:
    print("✅ Employee details retrieved successfully!")
else:
    print("❌ Failed to retrieve employee details!")

KeyboardInterrupt: 

### 4.4 Update Employee

Update employee information.

In [ ]:
# Update employee
employee_id = 1  # Change this to a valid employee ID
update_data = {
    "position": "Senior Software Engineer",
    "salary": "85000.00"
}

response = requests.patch(f"{BASE_URL}/employee/{employee_id}/", json=update_data, headers=HEADERS)
print_response(response, f"Update Employee {employee_id}")

if response.status_code == 200:
    print("✅ Employee updated successfully!")
else:
    print("❌ Failed to update employee!")

### 4.5 Delete Employee

Delete an employee record.

In [ ]:
# Delete employee (use with caution!)
# employee_id = 999  # Change this to a valid employee ID you want to delete

# response = requests.delete(f"{BASE_URL}/employee/{employee_id}/", headers=HEADERS)
# print_response(response, f"Delete Employee {employee_id}")

# if response.status_code == 204:
#     print("✅ Employee deleted successfully!")
# else:
#     print("❌ Failed to delete employee!")

print("⚠️ Delete employee endpoint commented out for safety. Uncomment to test.")

---
## 5. Client Management APIs <a id="clients"></a>

Test all client management endpoints.

### 5.1 Create Client

Create a new client record.

In [ ]:
# Create a new client
client_data = {
    "company_name": "Acme Corporation",
    "client_code": "ACME001",
    "contact_person": "Alice Johnson",
    "email": "alice@acmecorp.com",
    "phone": "+1234567890",
    "street": "456 Business Ave",
    "city": "San Francisco",
    "state": "CA",
    "zip_code": "94102",
    "country": "USA",
    "gstin": "22AAAAA0000A1Z5",
    "status": "Active",
    "contractStartDate": "2024-01-01T00:00:00Z",
    "contractEndDate": "2025-12-31T23:59:59Z"
}

response = requests.post(f"{BASE_URL}/clients/", json=client_data, headers=HEADERS)
print_response(response, "Create Client")

if response.status_code == 201:
    client = response.json()
    CLIENT_ID = client.get('id')
    print(f"✅ Client created successfully with ID: {CLIENT_ID}")
else:
    print("❌ Failed to create client!")

### 5.2 List All Clients

Get a list of all clients.

In [ ]:
# List all clients
response = requests.get(f"{BASE_URL}/clients/", headers=HEADERS)
print_response(response, "List All Clients")

if response.status_code == 200:
    clients = response.json()
    print(f"✅ Found {len(clients)} clients")
else:
    print("❌ Failed to retrieve clients!")

### 5.3 Get Client by ID

Retrieve details of a specific client.

In [ ]:
# Get client by ID
client_id = 1  # Change this to a valid client ID

response = requests.get(f"{BASE_URL}/clients/{client_id}/", headers=HEADERS)
print_response(response, f"Get Client {client_id}")

if response.status_code == 200:
    print("✅ Client details retrieved successfully!")
else:
    print("❌ Failed to retrieve client details!")

### 5.4 Update Client

Update client information.

In [ ]:
# Update client
client_id = 1  # Change this to a valid client ID
update_data = {
    "status": "Inactive",
    "contact_person": "Bob Smith"
}

response = requests.patch(f"{BASE_URL}/clients/{client_id}/", json=update_data, headers=HEADERS)
print_response(response, f"Update Client {client_id}")

if response.status_code == 200:
    print("✅ Client updated successfully!")
else:
    print("❌ Failed to update client!")

### 5.5 Delete Client

Delete a client record (cascades to related units).

In [ ]:
# Delete client (use with caution! This will also delete all related units)
# client_id = 999  # Change this to a valid client ID you want to delete

# response = requests.delete(f"{BASE_URL}/clients/{client_id}/", headers=HEADERS)
# print_response(response, f"Delete Client {client_id}")

# if response.status_code == 204:
#     print("✅ Client deleted successfully!")
# else:
#     print("❌ Failed to delete client!")

print("⚠️ Delete client endpoint commented out for safety. Uncomment to test.")

---
## 6. Client Unit APIs <a id="client-units"></a>

Test all client unit management endpoints.

### 6.1 Create Client Unit

Create a new client unit.

In [ ]:
# Create a new client unit
unit_data = {
    "client": 1,  # Change this to a valid client ID
    "unit_name": "Main Office",
    "unit_code": "ACME001-MAIN-001",
    "print_name": "Acme Corp - Main Office",
    "shipping_address": "456 Business Ave, San Francisco, CA 94102",
    "billing_name": "Acme Corporation",
    "billing_address": "456 Business Ave, San Francisco, CA 94102",
    "phone": "+1234567890",
    "email": "mainoffice@acmecorp.com",
    "no_of_employees": 50,
    "min_age": 18,
    "max_age": 65,
    "is_gst_applicable": True
}

response = requests.post(f"{BASE_URL}/clients/units/", json=unit_data, headers=HEADERS)
print_response(response, "Create Client Unit")

if response.status_code == 201:
    unit = response.json()
    UNIT_ID = unit.get('id')
    print(f"✅ Client unit created successfully with ID: {UNIT_ID}")
else:
    print("❌ Failed to create client unit!")

### 6.2 List All Client Units

Get a list of all client units.

In [ ]:
# List all client units
response = requests.get(f"{BASE_URL}/clients/units/", headers=HEADERS)
print_response(response, "List All Client Units")

if response.status_code == 200:
    units = response.json()
    print(f"✅ Found {len(units)} client units")
else:
    print("❌ Failed to retrieve client units!")

### 6.3 Get Client Unit by ID

Retrieve details of a specific client unit.

In [ ]:
# Get client unit by ID
unit_id = 1  # Change this to a valid unit ID

response = requests.get(f"{BASE_URL}/clients/units/{unit_id}/", headers=HEADERS)
print_response(response, f"Get Client Unit {unit_id}")

if response.status_code == 200:
    print("✅ Client unit details retrieved successfully!")
else:
    print("❌ Failed to retrieve client unit details!")

### 6.4 Update Client Unit

Update client unit information.

In [ ]:
# Update client unit
unit_id = 1  # Change this to a valid unit ID
update_data = {
    "no_of_employees": 75,
    "phone": "+1234567899"
}

response = requests.patch(f"{BASE_URL}/clients/units/{unit_id}/", json=update_data, headers=HEADERS)
print_response(response, f"Update Client Unit {unit_id}")

if response.status_code == 200:
    print("✅ Client unit updated successfully!")
else:
    print("❌ Failed to update client unit!")

### 6.5 Delete Client Unit

Delete a client unit.

In [ ]:
# Delete client unit (use with caution!)
# unit_id = 999  # Change this to a valid unit ID you want to delete

# response = requests.delete(f"{BASE_URL}/clients/units/{unit_id}/", headers=HEADERS)
# print_response(response, f"Delete Client Unit {unit_id}")

# if response.status_code == 204:
#     print("✅ Client unit deleted successfully!")
# else:
#     print("❌ Failed to delete client unit!")

print("⚠️ Delete client unit endpoint commented out for safety. Uncomment to test.")

---
## 7. Department Contact APIs <a id="department-contacts"></a>

Test all department contact management endpoints.

### 7.1 Create Department Contact

Create a new department contact.

In [ ]:
# Create a new department contact
contact_data = {
    "name": "Sarah Williams",
    "designation": "Accounts Manager",
    "phone": "+1234567890",
    "mobile": "+0987654321",
    "email": "sarah.williams@acmecorp.com"
}

response = requests.post(f"{BASE_URL}/clients/contacts/", json=contact_data, headers=HEADERS)
print_response(response, "Create Department Contact")

if response.status_code == 201:
    contact = response.json()
    CONTACT_ID = contact.get('id')
    print(f"✅ Department contact created successfully with ID: {CONTACT_ID}")
else:
    print("❌ Failed to create department contact!")

### 7.2 List All Department Contacts

Get a list of all department contacts.

In [ ]:
# List all department contacts
response = requests.get(f"{BASE_URL}/clients/contacts/", headers=HEADERS)
print_response(response, "List All Department Contacts")

if response.status_code == 200:
    contacts = response.json()
    print(f"✅ Found {len(contacts)} department contacts")
else:
    print("❌ Failed to retrieve department contacts!")

### 7.3 Get Department Contact by ID

Retrieve details of a specific department contact.

In [ ]:
# Get department contact by ID
contact_id = 1  # Change this to a valid contact ID

response = requests.get(f"{BASE_URL}/clients/contacts/{contact_id}/", headers=HEADERS)
print_response(response, f"Get Department Contact {contact_id}")

if response.status_code == 200:
    print("✅ Department contact details retrieved successfully!")
else:
    print("❌ Failed to retrieve department contact details!")

### 7.4 Update Department Contact

Update department contact information.

In [ ]:
# Update department contact
contact_id = 1  # Change this to a valid contact ID
update_data = {
    "designation": "Senior Accounts Manager",
    "mobile": "+1111111111"
}

response = requests.patch(f"{BASE_URL}/clients/contacts/{contact_id}/", json=update_data, headers=HEADERS)
print_response(response, f"Update Department Contact {contact_id}")

if response.status_code == 200:
    print("✅ Department contact updated successfully!")
else:
    print("❌ Failed to update department contact!")

### 7.5 Delete Department Contact

Delete a department contact.

In [ ]:
# Delete department contact (use with caution!)
# contact_id = 999  # Change this to a valid contact ID you want to delete

# response = requests.delete(f"{BASE_URL}/clients/contacts/{contact_id}/", headers=HEADERS)
# print_response(response, f"Delete Department Contact {contact_id}")

# if response.status_code == 204:
#     print("✅ Department contact deleted successfully!")
# else:
#     print("❌ Failed to delete department contact!")

print("⚠️ Delete department contact endpoint commented out for safety. Uncomment to test.")

---
## 8. Advanced Testing Scenarios

Additional test scenarios for filtering, searching, and pagination.

### 8.1 User Filtering and Search

Test filtering and search functionality for users.

In [ ]:
# Filter active users
response = requests.get(f"{BASE_URL}/users/?is_active=true", headers=HEADERS)
print_response(response, "Filter Active Users")

# Search users by name
response = requests.get(f"{BASE_URL}/users/?search=test", headers=HEADERS)
print_response(response, "Search Users by Name")

# Order users by creation date
response = requests.get(f"{BASE_URL}/users/?ordering=-created_at", headers=HEADERS)
print_response(response, "Order Users by Creation Date (Descending)")

### 8.2 Error Handling Tests

Test error scenarios and validation.

In [ ]:
# Test invalid login
invalid_login = {
    "email": "invalid@example.com",
    "password": "wrongpassword"
}
response = requests.post(f"{BASE_URL}/token/", json=invalid_login, headers=HEADERS)
print_response(response, "Invalid Login Test")

# Test accessing protected endpoint without token
no_auth_headers = {"Content-Type": "application/json"}
response = requests.get(f"{BASE_URL}/users/me/", headers=no_auth_headers)
print_response(response, "Access Protected Endpoint Without Token")

# Test creating user with invalid data
invalid_user = {
    "name": "Test",
    "email": "invalid-email",  # Invalid email format
    "password": "123"  # Too short
}
response = requests.post(f"{BASE_URL}/users/", json=invalid_user, headers=HEADERS)
print_response(response, "Create User with Invalid Data")

---
## Summary

This notebook covers comprehensive testing of all REST API endpoints in the Management System:

✅ **Authentication**: Login, Token Refresh, Token Verification

✅ **User Management**: CRUD operations, Profile, Password Change, Statistics

✅ **Employee Management**: CRUD operations for employee records

✅ **Client Management**: CRUD operations for client records

✅ **Client Unit Management**: CRUD operations for client units

✅ **Department Contact Management**: CRUD operations for department contacts

✅ **Advanced Features**: Filtering, Searching, Ordering, Error Handling

### Next Steps:
1. Make sure your Django backend is running on `http://localhost:8000`
2. Update the `BASE_URL` in the setup section if needed
3. Run the cells sequentially to test all endpoints
4. Modify the test data as needed for your specific use case

### Notes:
- Delete operations are commented out for safety
- Some endpoints require admin privileges
- Make sure to have valid IDs when testing specific resource endpoints
- Check the Swagger documentation at `http://localhost:8000/swagger/` for more details